# 01 — SoilGrids API Extraction
**Project:** Supervised Classification of Agricultural Soil Types in Togo  
**Author:** Daniel ESSONANI | Supervised by [M. Leri TCHANTCHO](https://www.linkedin.com/in/leri-damigouri-tchantcho-28503873/) 

**Date:** February 2026

---

## Overview

This notebook queries the **SoilGrids v2.0 REST API** (ISRIC/FAO) to extract the dominant WRB soil type  
for each of the **599 administrative cantons** of Togo, using their geographic centroids.

### What SoilGrids provides
SoilGrids is a global soil database built by ISRIC that predicts soil properties and types  
at 250m resolution worldwide, using machine learning trained on 200,000+ field observations.

### WRB Classification
The **World Reference Base for Soil Resources (WRB)** is the international standard for soil naming,  
used by FAO. Each query returns the dominant WRB class and probability scores for the top 3 classes.

---

## ⚠️ Known Issues & Solutions (encountered during this project)

| Issue | Cause | Solution applied |
|-------|-------|-----------------|
| Extraction stopped at canton 268 | Kernel timeout / unstable connection | Resume mechanism using `iloc[267:]` + pre-saved `resultats` list |
| `TypeError: 'NoneType' is not subscriptable` | API returned HTTP error, `.json()` call on empty response | Wrapped everything in `try/except`, added explicit status code check |
| Slow extraction (~1 canton/sec) | API rate limit + `time.sleep(1)` | Kept 1s delay; total ~10 min for 599 cantons — acceptable |
| Some centroids fall outside land | Border cantons near coast/river | API still returns a result (nearest land point); kept as-is |


## 1. Install & Import Dependencies

In [ ]:
# Install required libraries (run once)
# !pip install requests geopandas pandas

import requests
import geopandas as gpd
import pandas as pd
import time
import os

print("Libraries loaded ✅")
print(f"Working directory: {os.getcwd()}")


## 2. Load the Cantons Shapefile

The shapefile contains **599 cantons** of Togo with administrative attributes.  
We need the geometry to compute centroids (lon/lat) for each canton.

> **Adjust the path** below to match your local file location.


In [ ]:
# ── Load shapefile ──────────────────────────────────────────────────────────
SHAPEFILE_PATH = "data/cantons_togo.shp"   # <-- update this path

cantons = gpd.read_file(SHAPEFILE_PATH)

# Quick inspection
print(f"CRS (coordinate system): {cantons.crs}")
print(f"Number of cantons: {len(cantons)}")
print(f"Columns: {cantons.columns.tolist()}")
cantons.head(3)


## 3. Compute Centroids

We compute the geographic centroid (center point) of each canton polygon.  
This lon/lat pair is then used to query SoilGrids.

> **Note:** The CRS is EPSG:4326 (WGS84), so centroids are already in decimal degrees — no reprojection needed.


In [ ]:
# ── Compute centroids ────────────────────────────────────────────────────────
cantons["centroid"] = cantons.geometry.centroid
cantons["lon"] = cantons["centroid"].x
cantons["lat"] = cantons["centroid"].y

print("Centroid sample:")
print(cantons[["CANTON", "lon", "lat"]].head(5))


## 4. Test the SoilGrids API

Before running the full extraction, we test connectivity with a single point  
at the approximate center of Togo (lon=1.0, lat=8.5).

Expected response includes:
- `wrb_class_name`: dominant WRB soil type
- `wrb_class_probability`: list of [soil_name, probability_%] for top 3 classes


In [ ]:
# ── API connection test ───────────────────────────────────────────────────────
API_URL = "https://rest.isric.org/soilgrids/v2.0/classification/query"

test_params = {"lon": 1.0, "lat": 8.5, "number_classes": 3}
resp = requests.get(API_URL, params=test_params, timeout=15)

print(f"HTTP Status: {resp.status_code}")
if resp.status_code == 200:
    data = resp.json()
    print(f"Dominant soil: {data['wrb_class_name']}")
    print(f"Top 3 probabilities: {data['wrb_class_probability']}")
    print("\n✅ API is reachable and returning data!")
else:
    print("❌ API call failed. Check your connection or the API status at https://rest.isric.org")


## 5. Define the Extraction Function

This function queries SoilGrids for a single (lon, lat) point and returns  
the dominant WRB soil type + top 2 secondary types with probabilities.

### Error handling
- `timeout=15`: avoids infinite hangs on slow connections
- `try/except`: catches network errors, JSON decode errors, etc.
- Returns `"ERROR"` dict on failure so the loop never crashes


In [ ]:
# ── Extraction function ───────────────────────────────────────────────────────
def get_soil_type(lon: float, lat: float) -> dict:
    """
    Query SoilGrids v2.0 API for the WRB soil classification at a given point.

    Parameters
    ----------
    lon : float
        Longitude in decimal degrees (WGS84)
    lat : float
        Latitude in decimal degrees (WGS84)

    Returns
    -------
    dict with keys:
        - soil_dominant     : dominant WRB class name
        - soil_prob1        : name of most probable WRB class
        - soil_prob1_pct    : probability (%) of most probable class
        - soil_prob2        : name of second most probable WRB class
        - soil_prob2_pct    : probability (%) of second most probable class
    """
    API_URL = "https://rest.isric.org/soilgrids/v2.0/classification/query"
    params  = {"lon": lon, "lat": lat, "number_classes": 3}

    try:
        response = requests.get(API_URL, params=params, timeout=15)

        if response.status_code == 200:
            data = response.json()
            probs = data.get("wrb_class_probability", [])
            return {
                "soil_dominant":  data.get("wrb_class_name", "Unknown"),
                "soil_prob1":     probs[0][0] if len(probs) > 0 else None,
                "soil_prob1_pct": probs[0][1] if len(probs) > 0 else None,
                "soil_prob2":     probs[1][0] if len(probs) > 1 else None,
                "soil_prob2_pct": probs[1][1] if len(probs) > 1 else None,
            }
        else:
            print(f"\n⚠️  HTTP {response.status_code} for ({lon:.4f}, {lat:.4f})")
            return {"soil_dominant": "ERROR", "soil_prob1": None,
                    "soil_prob1_pct": None, "soil_prob2": None, "soil_prob2_pct": None}

    except Exception as e:
        print(f"\n⚠️  Exception for ({lon:.4f}, {lat:.4f}): {e}")
        return {"soil_dominant": "ERROR", "soil_prob1": None,
                "soil_prob1_pct": None, "soil_prob2": None, "soil_prob2_pct": None}


## 6. Full Extraction — All 599 Cantons

### ⚠️ Resume mechanism
During this project, the extraction **crashed at canton 268** due to a kernel timeout.  
The resume logic below lets you restart from any canton without losing previous results:

1. If `sols_togo_cantons.csv` already exists → load it and resume from where it stopped
2. Otherwise → start fresh from canton 0

**Estimated time:** ~10 minutes for 599 cantons (1 second delay per request to respect API limits)


In [ ]:
# ── Resume-safe extraction ────────────────────────────────────────────────────
OUTPUT_CSV = "sols_togo_cantons.csv"
SLEEP_BETWEEN_REQUESTS = 1  # seconds — do not lower this (API rate limit)

# Check if a partial extraction already exists
if os.path.exists(OUTPUT_CSV):
    df_existing = pd.read_csv(OUTPUT_CSV)
    already_done = set(df_existing["OBJECTID"].tolist())
    resultats = df_existing.to_dict("records")
    print(f"↩️  Resuming — {len(already_done)} cantons already extracted.")
else:
    already_done = set()
    resultats = []
    print("🚀 Starting fresh extraction...")

# Filter to only cantons not yet processed
cantons_todo = cantons[~cantons["OBJECTID"].isin(already_done)].copy()
total = len(cantons_todo)
print(f"Remaining to process: {total} cantons\n")

# ── Main extraction loop ──────────────────────────────────────────────────────
for idx, (_, row) in enumerate(cantons_todo.iterrows()):
    print(f"  [{idx+1}/{total}] {row['CANTON']}...", end="\r")

    soil = get_soil_type(row["lon"], row["lat"])
    soil["OBJECTID"] = int(row["OBJECTID"])
    soil["CANTON"]   = row["CANTON"]
    resultats.append(soil)

    # Save checkpoint every 50 cantons (protects against crashes)
    if (idx + 1) % 50 == 0:
        pd.DataFrame(resultats).to_csv(OUTPUT_CSV, index=False)
        print(f"\n  💾 Checkpoint saved at canton {idx+1}")

    time.sleep(SLEEP_BETWEEN_REQUESTS)

# Final save
df_sols = pd.DataFrame(resultats)
df_sols.to_csv(OUTPUT_CSV, index=False)
print(f"\n\n✅ Extraction complete! {len(df_sols)} cantons saved to '{OUTPUT_CSV}'")
df_sols.head()


## 7. Quick Results Summary

In [ ]:
# ── Distribution of WRB soil types ───────────────────────────────────────────
print("=== WRB Soil Type Distribution (599 cantons) ===\n")
dist = df_sols["soil_dominant"].value_counts()
print(dist.to_string())

# Error check
errors = df_sols[df_sols["soil_dominant"] == "ERROR"]
print(f"\n⚠️  Extraction errors: {len(errors)} cantons")
if len(errors) > 0:
    print("   → Re-run this notebook to retry failed cantons (resume mechanism will handle it)")
